# A 3D part from CAD, meshed for FEM

Pipeline (one section per step):

| Step | What | Where |
|---|---|---|
| 1 | CAD: 90° bent pipe with straight legs, every dimension a parameter | `bent_pipe.py` (FreeCAD, headless) → `pipe.step` |
| 2 | Tet mesh + named boundary groups + checks against the CAD | `mesh_pipe.py` (gmsh) → `pipe.msh` |
| 3 | Look at it: boundary groups, then a cut through the wall | gmsh GUI |
| 4 | FEM: steady heat conduction in the wall (next) | — |

The wall is the domain. Mesh contract the solver reads: metres, gmsh 4.1, 4-node tets, volume group `wall`,
surface groups `inner`, `outer`, `end_a`, `end_b`.

```
        end_b
   ==============╮      bend: centreline radius R_BEND, 90°
                 ║
                 ║  leg a
                 ║
               end_a
```

## Step 1 — CAD in FreeCAD

Same idea as week 2's Step 1: no typed-in coordinates, only parameters (`R_OUT`, `R_IN`, `R_BEND`, `L` at the top
of `bent_pipe.py`). The pipe is (outer tube) − (inner tube), each tube = cylinder + quarter torus + cylinder.

The script asserts its volume against Pappus: annulus area × centreline length,
$V = \pi (r_o^2 - r_i^2)\,(2L + \tfrac{\pi}{2} R)$.

FreeCAD writes **mm**; gmsh rescales to metres on import.

In [ ]:
!/Applications/FreeCAD.app/Contents/Resources/bin/freecadcmd bent_pipe.py | tail -1

## Step 2 — Tet mesh with named boundaries

STEP keeps the solid but not face names, so `mesh_pipe.py` tags faces **by shape**, not by tag number (numbers shift
whenever the CAD changes). Largest principal curvature $\kappa$ of each face:

| face | $\kappa$ | group |
|---|---|---|
| flat annulus | 0 | `end_a` (lower $y$) / `end_b` |
| cylinder or torus wall, radius $r$ | $1/r$ | larger $\kappa$ → `inner`, smaller → `outer` |

The meshing itself lives in `step_to_msh.py`, which works on **any** STEP file (`python step_to_msh.py part.step [h]`):
convert units to metres, heal dirty CAD, make touching solids share faces, mesh, check. `mesh_pipe.py` is that plus
the pipe-specific tagging.

Size: $h = \min(h_{\max},\ 2\pi r / 20)$, capped by $h_{\max}$ (default 5 mm = 2 cells through the 10 mm wall), while
curvature sizing gives small holes and fillets at least 20 cells per full circle. For the pipe $2\pi r/20 \approx 13$ mm,
so $h_{\max}$ wins everywhere. Here we *want* a second size source, unlike week 1. Pass a different $h_{\max}$ in metres
for a convergence study.

Checks against the exact CAD: total volume and every face's area. Flat facets cut corners on a curved face, so the mesh
comes out short by about $(2\pi/20)^2/6$ at worst; the script allows 3× that.

In [1]:
import sys

!{sys.executable} mesh_pipe.py

radii from curvature: inner 40.0 mm, outer 50.0 mm
group          CAD        mesh  rel err
wall    2.3627e-03  2.3627e-03 2.02e-05
inner   2.1001e-01  2.0991e-01 4.83e-04
outer   2.6252e-01  2.6244e-01 3.09e-04
end_a   2.8274e-03  2.8271e-03 1.09e-04
end_b   2.8274e-03  2.8271e-03 1.09e-04
h = 5 mm: 29647 nodes, 110295 tets, worst minSICN 0.295 (1 = perfect, <= 0 = inverted)


## Step 3 — Look at it

Boundary groups first: <span style="color:#dc3c32">inner</span>, <span style="color:#aaaaaa">outer</span>,
<span style="color:#286edc">end_a</span>, <span style="color:#28aa50">end_b</span>.

In [2]:
import sys

import gmsh

from step_to_msh import fresh_gmsh  # one gmsh per kernel: never finalize between views

GROUP_COLORS = {"inner": (220, 60, 50), "outer": (170, 170, 170), "end_a": (40, 110, 220), "end_b": (40, 170, 80)}


def color_groups():
    # Paint each surface by its physical group; Mesh.ColorCarousel = 1 then draws the mesh in entity colours.
    for dim, tag in gmsh.model.getPhysicalGroups(2):
        entities = gmsh.model.getEntitiesForPhysicalGroup(dim, tag)
        gmsh.model.setColor([(dim, int(e)) for e in entities], *GROUP_COLORS[gmsh.model.getPhysicalName(dim, tag)])


fresh_gmsh()
gmsh.open("pipe.msh")
color_groups()

DISPLAY = {
    "Mesh.SurfaceFaces": 1,   # shaded triangles                        (default 0)
    "Mesh.SurfaceEdges": 1,   # triangle wireframe                      (default 1)
    "Mesh.ColorCarousel": 1,  # colour by entity, i.e. by setColor above (default 1)
    "Geometry.Points": 0,     # hide the CAD entities drawn on top of the mesh
    "Geometry.Curves": 0,
    "Geometry.Surfaces": 0,
    "General.Trackball": 0,   # so the two rotations below are plain Euler angles
    "General.RotationX": -60,
    "General.RotationZ": 20,
}
for option, value in DISPLAY.items():
    gmsh.option.setNumber(option, value)

# Tools > Visibility shows/hides one group at a time.
if 'close' not in sys.argv:
    gmsh.fltk.run()

Info    : Reading 'pipe.msh'...
Info    : 39 entities
Info    : 29647 nodes
Info    : 156033 elements
Info    : Done reading 'pipe.msh'                                                                                
-------------------------------------------------------
Version       : 4.15.2
License       : GNU General Public License
Build OS      : MacOSARM-sdk
Build date    : 20260324
Build host    : gmsh.info
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blas[petsc] Blossom Cgns DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack Lapack[petsc] MathEx[contrib] Med Mesh Metis[contrib] Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom PETSc Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] TouchBar Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.4.0
PETSc version : 3.17.4 (real arithmtic)
OCC version   : 7.8.1
MED version   : 4.1.0
Package

The surface view hides the tets. A **clipping plane** through the pipe's mid-plane $z = 0$ keeps only the
$z \le 0$ half (plane $Ax + By + Cz + D \ge 0$ with $C = -1$), so the volume tets show up in cross-section: the
green band is the wall, two tets thick at $h$ = 5 mm. That is thin for the bend; it's what a convergence study
on $h$ will tell us.

In [4]:
fresh_gmsh()
gmsh.open("pipe.msh")
color_groups()

DISPLAY_CUT = {
    "Mesh.SurfaceFaces": 1,
    "Mesh.VolumeFaces": 1,    # draw the tets themselves                (default 0)
    "Mesh.VolumeEdges": 1,
    "Mesh.ColorCarousel": 1,
    "Geometry.Points": 0,
    "Geometry.Curves": 0,
    "Geometry.Surfaces": 0,
    "General.Clip0A": 0,      # clipping plane 0: 0x + 0y - z + 0 >= 0 keeps z <= 0
    "General.Clip0B": 0,
    "General.Clip0C": -1,
    "General.Clip0D": 0,
    "Mesh.Clip": 1,           # bitmask: apply plane 0 to the mesh
}
for option, value in DISPLAY_CUT.items():
    gmsh.option.setNumber(option, value)

# Tools > Clipping moves the plane live.
if 'close' not in sys.argv:
    gmsh.fltk.run()

Info    : Reading 'pipe.msh'...
Info    : 39 entities
Info    : 29647 nodes
Info    : 156033 elements
Info    : Done reading 'pipe.msh'                                                                                
-------------------------------------------------------
Version       : 4.15.2
License       : GNU General Public License
Build OS      : MacOSARM-sdk
Build date    : 20260324
Build host    : gmsh.info
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blas[petsc] Blossom Cgns DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack Lapack[petsc] MathEx[contrib] Med Mesh Metis[contrib] Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom PETSc Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] TouchBar Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.4.0
PETSc version : 3.17.4 (real arithmtic)
OCC version   : 7.8.1
MED version   : 4.1.0
Package

## Bonus — any STEP file

`step_to_msh.py` has no pipe in it, so it should mesh parts nobody drew for us. Three downloaded ones live in
`samples/` (sources in `samples/SOURCES.md`):

| file | what | lesson |
|---|---|---|
| `screw.step` | small screw, 10 faces | easy case |
| `linkrods.step` | 5 mm part, 37 faces (B-spline, torus, cylinder, plane) | $h_{\max}$ = 5 mm is bigger than the part: curvature sizing does all the work. Delaunay left a sliver (minSICN 0.028) until the Netgen optimiser was switched on |
| `t20_data.step` | 42 mm part, all 21 faces B-spline | at 5 mm, thin strips get ~6 triangles and miss their CAD area by 11% (the script warns); 1 mm resolves them (0.6%) |

Each CAD face gets its own colour, so you can see where one face ends and the next begins. Close the window to move
on to the next part.

In [5]:
from step_to_msh import mesh_step

SAMPLES = {  # file: h_max in metres
    "samples/screw.step": 0.005,
    "samples/linkrods.step": 0.005,
    "samples/t20_data.step": 0.001,
}
DISPLAY_FACES = {
    "Mesh.SurfaceFaces": 1,
    "Mesh.SurfaceEdges": 1,
    "Mesh.ColorCarousel": 1,  # colour by entity: one colour per CAD face
    "Geometry.Points": 0,
    "Geometry.Curves": 0,
    "General.Trackball": 0,
    "General.RotationX": -60,
    "General.RotationZ": 30,
}

for step, h_max in SAMPLES.items():
    mesh_step(step, h_max)  # prints its checks; also resets gmsh, so set the display options after it
    for option, value in DISPLAY_FACES.items():
        gmsh.option.setNumber(option, value)
    if 'close' not in sys.argv:
        gmsh.fltk.run()

Info    : Clearing all models and views...
Info    : Done clearing all models and views
screw.step: 1 volumes, 10 faces, volume 3.7408e-06 m^3 (CAD 3.7883e-06), worst face area err 1.6e-02
h_max = 5 mm: 1173 nodes, 4262 tets, worst minSICN 0.347 (1 = perfect, <= 0 = inverted)
linkrods.step: 1 volumes, 37 faces, volume 3.8632e-09 m^3 (CAD 3.8470e-09), worst face area err 3.3e-02
h_max = 5 mm: 33086 nodes, 140828 tets, worst minSICN 0.376 (1 = perfect, <= 0 = inverted)
t20_data.step: 1 volumes, 21 faces, volume 1.8394e-05 m^3 (CAD 1.8388e-05), worst face area err 5.8e-03
h_max = 1 mm: 17856 nodes, 82052 tets, worst minSICN 0.074 (1 = perfect, <= 0 = inverted)
